# Caso A — Abastecimiento · Modelamiento desplegado

> **Qué hace este notebook.** Reproduce, paso a paso y a la vista, el mismo modelamiento que corre en producción (`tostao_ml.cases.caso_a.run_case_a`): features, partición temporal, optimización de hiperparámetros, ajuste del modelo cuantílico, evaluación, comparación de modelos y optimización de pedido.

> **Por qué desplegado.** En producción el modelo vive en un módulo y el pipeline lo invoca; aquí se abre para que se vea cada decisión de modelamiento. Reutiliza las mismas piezas del framework, así que los resultados coinciden.

> **Cómo ejecutar.** `Restart & Run All`; es determinista (semilla fija).

## 1. Datos: tabla maestra semanal

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_d, _ = masters.build_master_a(
        catalog.load('a_ventas_historicas'), catalog.load('a_catalogo_productos'),
        catalog.load('a_maestro_tiendas'), catalog.load('a_inventario_actual'),
        catalog.load('a_ground_truth_trends'))
weekly = masters.aggregate_weekly_a(master_d)
print('Master semanal:', weekly.shape)

## 2. Feature engineering (transformadores del framework)

Rezagos y media rodante *group-aware* por SKU-tienda (sin filtrar el valor actual), estacionalidad cíclica de la semana y codificación de frecuencia de las categóricas. Es la misma `build_features_a`, aquí desplegada.

In [ ]:
from tostao_ml.framework.io import set_global_seed
from tostao_ml.framework.features import GroupLagFeatures, CyclicalEncoder, FrequencyEncoder
from tostao_ml.cases.caso_a import FEATURES, GROUP, TARGET

set_global_seed(42)
df = weekly.sort_values([*GROUP, 'semana']).copy()
df = GroupLagFeatures(GROUP, 'semana', TARGET, lags=(1, 2), rolling_windows=(3,)).fit_transform(df)
df = CyclicalEncoder(['semana'], {'semana': 52}, drop_original=False).fit_transform(df)
df = FrequencyEncoder(['categoria', 'ciudad', 'trend_type'], drop_original=False).fit_transform(df)
features = df.dropna(subset=[f'{TARGET}_lag_1']).reset_index(drop=True)
print('Features usadas:', FEATURES)
features[FEATURES].head()

## 3. Partición temporal (holdout sin fuga de futuro)

Las últimas 3 semanas son test; el resto, entrenamiento. Nunca se usa el futuro para predecir el pasado.

In [ ]:
test_weeks = 3
cutoff = features['semana'].max() - test_weeks
train = features[features['semana'] <= cutoff].copy()
test = features[features['semana'] > cutoff].copy()
print(f'train={train.shape}  test={test.shape}  corte=semana {cutoff}')

## 4. Optimización de hiperparámetros (Optuna, walk-forward)

TPE sobre `learning_rate`, `max_depth`, `max_iter`, validando con `TimeSeriesSplit` (respeta el orden temporal) y optimizando el WAPE.

In [ ]:
from tostao_ml.framework.evaluation import metrics, time_series_splitter
from tostao_ml.framework.tuning import tune_model

HPO_SPACE = {
    'learning_rate': {'type': 'float', 'low': 0.02, 'high': 0.3, 'log': True},
    'max_depth': {'type': 'int', 'low': 2, 'high': 8},
    'max_iter': {'type': 'int', 'low': 80, 'high': 350},
}
ordered = train.sort_values('semana')
tuning = tune_model(
    'gbr', HPO_SPACE, ordered[FEATURES], ordered[TARGET],
    scorer=lambda m, xv, yv: metrics.wape(yv, m.predict(xv)),
    splitter=time_series_splitter(n_splits=3), direction='minimize',
    n_trials=20, sampler='tpe', pruner='none', seed=42)
hp = {k: tuning.best_params[k] for k in ('learning_rate', 'max_depth', 'max_iter')}
print('Mejor configuración:', hp)

## 5. El modelo: gradient boosting cuantílico (pérdida pinball)

Un estimador por cuantil (0.1 / 0.5 / 0.9). Los cuantiles dan **intervalos** de predicción, insumo del optimizador de pedido.

In [ ]:
from tostao_ml.framework.models import QuantileGBRModel

quantiles = (0.1, 0.5, 0.9)
model = QuantileGBRModel(quantiles=quantiles, random_state=42, **hp)
model.fit(train[FEATURES], train[TARGET])
model

## 6. Predicción e intervalos sobre el holdout

In [ ]:
q_pred = model.predict_quantiles(test[FEATURES])
interval = model.predict_interval(test[FEATURES], coverage=quantiles[-1] - quantiles[0])
test = test.assign(pred=interval['median'], lower=interval['lower'], upper=interval['upper'])
test[[TARGET, 'pred', 'lower', 'upper']].head()

## 7. Métricas de desempeño (lectura, no definición)

WAPE (error porcentual robusto), R² (varianza explicada), y PICP/MPIW para la calidad de los intervalos, frente al baseline ingenuo de persistencia.

In [ ]:
import pandas as pd

y = test[TARGET].to_numpy(float)
naive = test[f'{TARGET}_lag_1'].to_numpy(float)
report = {
    **metrics.regression_report(y, test['pred']),
    'wape_naive': metrics.wape(y, naive),
    'picp': metrics.picp(y, test['lower'], test['upper']),
    'mpiw': metrics.mpiw(test['lower'], test['upper']),
}
pd.DataFrame([report]).T.rename(columns={0: 'valor'}).round(4)

## 8. Validación de varios modelos (más de un modelo)

Se enfrentan Ridge, GBR, el cuantílico (mediana) y un ensemble en el holdout. Ridge no tolera NaN de los rezagos iniciales, así que se imputan a 0 para una comparación justa.

In [ ]:
from tostao_ml.framework.models import RidgeRegressionModel, GBRRegressionModel, AveragingEnsemble
from tostao_ml.framework.evaluation import compare_models

gbr_hp = {k: hp[k] for k in ('learning_rate', 'max_depth', 'max_iter')}
candidatos = {
    'Ridge (lineal)': RidgeRegressionModel(random_state=42),
    'GBR': GBRRegressionModel(random_state=42, **gbr_hp),
    'Cuantílico (mediana)': model,
    'Ensemble (Ridge+GBR)': AveragingEnsemble([
        ('ridge', RidgeRegressionModel(random_state=42)),
        ('gbr', GBRRegressionModel(random_state=42, **gbr_hp))]),
}
comparison = compare_models(candidatos, train[FEATURES].fillna(0.0), train[TARGET],
                            test[FEATURES].fillna(0.0), test[TARGET], sort_by='wape')
comparison.round(4)

## 9. Figuras de desempeño e importancia de features

In [ ]:
from tostao_ml.framework.evaluation import performance
from tostao_ml.framework.interpret import permutation_importance, permutation_bar

performance.pred_vs_actual(test[TARGET], test['pred']).show()
performance.residuals_vs_pred(test[TARGET], test['pred']).show()
performance.model_comparison_bar(comparison['wape'].to_dict(), 'WAPE (menor es mejor)').show()
imp = permutation_importance(model, test[FEATURES], test[TARGET], metrics.rmse, n_repeats=3)
permutation_bar(imp).show()

## 10. De la predicción a la decisión: optimización de pedido (newsvendor)

Con los cuantiles se calcula el fractil crítico `Cu/(Cu+Co)` por SKU-tienda y se decide la cantidad que minimiza el costo esperado de faltante+sobrante, descontando el stock actual.

In [ ]:
import numpy as np
from tostao_ml.framework.optimization import NewsvendorPolicy, expected_cost

cu = (test['precio_venta'] - test['costo_unitario']).to_numpy(float)
co = test['costo_almacenamiento_semanal'].to_numpy(float)
stock = test['stock_actual'].to_numpy(float)
records, cost_model, cost_naive = [], 0.0, 0.0
for i in range(len(test)):
    qmap = {q: float(q_pred.iloc[i][f'q{q}']) for q in quantiles}
    dec = NewsvendorPolicy(cu=float(cu[i]), co=float(co[i])).order(qmap, float(stock[i]))
    records.append(dec)
    cost_model += expected_cost(dec['order_qty'], stock[i], np.array([y[i]]), cu[i], co[i])
    nq = max(0.0, naive[i] - stock[i])
    cost_naive += expected_cost(nq, stock[i], np.array([y[i]]), cu[i], co[i])
orders = pd.DataFrame(records, index=test.index)
ahorro = 100 * (cost_naive - cost_model) / cost_naive if cost_naive else 0.0
print(f'costo óptimo={cost_model:,.0f}  ingenuo={cost_naive:,.0f}  ahorro={ahorro:.1f}%')
performance.model_comparison_bar({'Política óptima': cost_model, 'Política ingenua': cost_naive},
                                 'costo esperado (menor es mejor)').show()
orders.head(15).round(2)

## Conclusión

El boosting cuantílico supera al baseline de persistencia y sus cuantiles alimentan una política newsvendor que reduce el costo esperado. Los intervalos aún sub-cubren, por lo que conviene calibrarlos antes de fijar niveles de servicio. Este es el mismo modelamiento que persiste el pipeline en `data/06_models/modelo_caso_a.pkl`.